# UC-ASSET-3 — Search Assets (catalog / collection / cross-catalog)

**Persona:** Scientist / Platform Auditor

**Goal:** Search assets by filter criteria (role, asset type) at the catalog
tier or within a single collection, follow offset-based pagination to retrieve
all pages, and demonstrate the cross-catalog search endpoint that requires
sysadmin credentials.

**Search API surface (path-scoped, routing-aware):**
- `POST /assets/catalogs/{catalog_id}/assets-search` — catalog-tier assets
  (those bound to no collection). Add `?all_collections=true` to span every
  collection under the catalog *plus* the catalog tier in one call.
- `POST /assets/catalogs/{catalog_id}/collections/{collection_id}/assets-search` — one collection
- `POST /assets/assets-search` — cross-catalog (sysadmin only); also accepts
  `?all_collections=true` to span every collection in each catalog.

Scope comes from the **path** (plus the `all_collections` flag), not the body.
Each endpoint resolves the asset `SEARCH` driver from routing config and falls
back to the `READ` driver when no dedicated search backend (e.g. Elasticsearch)
is pinned — so the same request shape works on any configured routing.

**Request body schema (`SearchQuery`):**
```json
{
  "filters": [{"field": "asset_type", "op": "eq", "value": "RASTER"}],
  "limit": 50,
  "offset": 0
}
```

Collection scope is tri-state on the catalog endpoint:
- default → catalog-tier assets only;
- `?all_collections=true` → every asset under the catalog (all collections + tier);
- to target a single collection's column from the catalog endpoint, add
  `{"field": "collection_id", "op": "eq", "value": "..."}` to `filters`.

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`,
`DYNASTORE_SYSADMIN_TOKEN`, `CATALOG_ID`, `COLLECTION_ID`

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL        = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
TOKEN           = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
SYSADMIN_TOKEN  = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
CATALOG_ID      = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID   = os.environ.get("COLLECTION_ID", "demo-collection")

PAGE_SIZE = 50

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}
sysadmin_client = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=120.0)

print(f"Connected to {BASE_URL}  catalog={CATALOG_ID}  collection={COLLECTION_ID}")

In [ ]:
# Catalog-scoped search: RASTER assets
# SearchQuery uses a list of AssetFilter objects:
#   field: dotted path into asset fields (asset_id, uri, asset_type, metadata.key)
#   op: eq | ne | contains | gt | lt | gte | lte
#   value: filter value

search_payload = {
    "filters": [
        {"field": "asset_type", "op": "eq", "value": "RASTER"},
    ],
    "limit": PAGE_SIZE,
    "offset": 0,
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/assets-search",
    content=json.dumps(search_payload),
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

result = r.json()
# result may be a list or a dict with 'assets' + pagination fields
if isinstance(result, list):
    assets_page = result
    total = None
else:
    assets_page = result.get("assets", result.get("items", []))
    total = result.get("total")

print(f"\nPage 1: {len(assets_page)} asset(s)  total={total}")
for a in assets_page[:5]:
    print(f"  asset_id={a.get('asset_id'):<30}  type={a.get('asset_type')}")

In [ ]:
# Pagination: iterate using offset until all pages exhausted
# The assets-search endpoint uses offset/limit pagination (not cursor tokens).
# Increment offset by limit until the returned page is shorter than the page size.

all_assets = list(assets_page)  # seed with page 1
offset = PAGE_SIZE

while len(assets_page) == PAGE_SIZE:
    search_payload["offset"] = offset
    r = client.post(
        f"/assets/catalogs/{CATALOG_ID}/assets-search",
        content=json.dumps(search_payload),
    )
    assert r.status_code == 200, f"Pagination failed at offset={offset}: {r.status_code}: {r.text}"
    page_result = r.json()
    assets_page = (
        page_result if isinstance(page_result, list)
        else page_result.get("assets", page_result.get("items", []))
    )
    all_assets.extend(assets_page)
    print(f"  offset={offset:5d}  page_size={len(assets_page):3d}  cumulative={len(all_assets)}")
    offset += PAGE_SIZE

print(f"\nTotal RASTER assets retrieved: {len(all_assets)}")
if total is not None:
    assert len(all_assets) == total or total == 0, (
        f"Expected {total} assets, got {len(all_assets)}"
    )

In [ ]:
# Collection-scoped search: same body, collection comes from the PATH
# POST /assets/catalogs/{catalog_id}/collections/{collection_id}/assets-search
# Returns only assets bound to COLLECTION_ID (the catalog endpoint above
# returns catalog-tier assets, i.e. those NOT bound to a collection).

collection_payload = {
    "filters": [
        {"field": "asset_type", "op": "eq", "value": "RASTER"},
    ],
    "limit": PAGE_SIZE,
    "offset": 0,
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/assets-search",
    content=json.dumps(collection_payload),
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

col_result = r.json()
col_assets = (
    col_result if isinstance(col_result, list)
    else col_result.get("assets", col_result.get("items", []))
)
print(f"\nCollection {COLLECTION_ID}: {len(col_assets)} RASTER asset(s)")
for a in col_assets[:5]:
    print(f"  asset_id={a.get('asset_id'):<30}  collection={a.get('collection_id')}")

In [ ]:
# Catalog-wide search: same catalog endpoint + ?all_collections=true
# Spans EVERY asset under the catalog (all collections + the catalog tier),
# not just the catalog-tier rows the default catalog search returns.

catalog_wide_payload = {
    "filters": [
        {"field": "asset_type", "op": "eq", "value": "RASTER"},
    ],
    "limit": PAGE_SIZE,
    "offset": 0,
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/assets-search?all_collections=true",
    content=json.dumps(catalog_wide_payload),
)
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

wide_result = r.json()
wide_assets = (
    wide_result if isinstance(wide_result, list)
    else wide_result.get("assets", wide_result.get("items", []))
)
print(f"\nCatalog-wide (all collections): {len(wide_assets)} RASTER asset(s)")
# Catalog-wide is a superset of catalog-tier-only (it also includes
# collection-bound assets), so it must return at least as many rows.
print("  (superset of the catalog-tier-only result above)")
for a in wide_assets[:5]:
    print(f"  asset_id={a.get('asset_id'):<30}  collection={a.get('collection_id')}")

In [ ]:
# Cross-catalog search: POST /assets/assets-search (sysadmin only)
# No catalog_id in path — scans all catalogs the caller can access.

global_payload = {
    "filters": [
        {"field": "asset_type", "op": "eq", "value": "RASTER"},
    ],
    "limit": PAGE_SIZE,
    "offset": 0,
}

r = sysadmin_client.post(
    "/assets/assets-search",
    content=json.dumps(global_payload),
)
print(r.status_code, r.text[:600])

if r.status_code == 200:
    global_result = r.json()
    global_assets = (
        global_result if isinstance(global_result, list)
        else global_result.get("assets", global_result.get("items", []))
    )
    global_total = global_result.get("total") if isinstance(global_result, dict) else None
    print(f"\nGlobal search returned {len(global_assets)} assets on page 1  total={global_total}")
elif r.status_code == 403:
    print("NOTE: global assets-search returned 403 even for sysadmin — platform may require "
          "a different privilege scope for cross-catalog search. Skipping global search assertions.")
    global_assets = []
    global_total = None
else:
    assert r.status_code == 200, f"Expected 200 for sysadmin global search, got {r.status_code}: {r.text}"


## Edge cases

### Global search with regular token → 403

The cross-catalog `/assets/assets-search` endpoint requires sysadmin scope.
A regular catalog-scoped token must receive `HTTP 403 Forbidden`.

In [ ]:
# Regular (non-sysadmin) token against global endpoint → 403
r = client.post(
    "/assets/assets-search",
    content=json.dumps(global_payload),
)
print(r.status_code, r.text[:300])
assert r.status_code in (401, 403), (
    f"Expected 401 or 403 for regular token on global search, got {r.status_code}: {r.text}"
)
print(f"{r.status_code} confirmed — global search denied for non-sysadmin token.")

In [ ]:
# metadata JSONB field search — performance note
#
# Filtering on nested metadata keys (e.g. metadata.sensor) uses a JSONB containment
# operator on the PostgreSQL `assets.metadata` column.
# There is currently NO GIN index on this column.
#
# For catalogs with > ~100k assets this becomes a sequential scan and will
# degrade query latency significantly.
#
# Recommended DDL (run as platform admin, per tenant schema):
#   CREATE INDEX CONCURRENTLY idx_assets_metadata_gin
#     ON {catalog_id}.assets USING gin(metadata jsonb_path_ops);
#
# Until this index is added, restrict metadata filters to small catalogs or
# combine them with asset_type / catalog_id filters to reduce scan size.

metadata_payload = {
    "filters": [
        {"field": "metadata.sensor", "op": "eq", "value": "OLI-2"},
    ],
    "limit": 10,
    "offset": 0,
}

r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/assets-search",
    content=json.dumps(metadata_payload),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

meta_result = r.json()
meta_assets = (
    meta_result if isinstance(meta_result, list)
    else meta_result.get("assets", meta_result.get("items", []))
)
print(f"metadata.sensor=OLI-2 → {len(meta_assets)} asset(s)")
print("WARN: no GIN index on metadata column — add idx_assets_metadata_gin for production scale.")

In [ ]:
# Soft-deleted assets are excluded from search by default
#
# When an asset is soft-deleted (DELETE without force=true), the `status`
# column is flipped to 'deleted'.  The assets-search implementation adds
# WHERE status <> 'deleted' to all queries, so soft-deleted assets are
# invisible to normal searches.
#
# To inspect soft-deleted assets an admin must query the DB directly:
#   SELECT * FROM {catalog_id}.assets WHERE status = 'deleted';
#
# Hard-deletion (force=true) removes the row entirely — not recoverable.

print("Soft-delete behaviour:")
print("  DELETE /assets/catalogs/{id}/assets/{aid}           → soft-delete (status='deleted')")
print("  DELETE /assets/catalogs/{id}/assets/{aid}?force=true → hard-delete (row removed)")
print("  assets-search always filters WHERE status <> 'deleted' (soft-deleted invisible)")

client.close()
sysadmin_client.close()
